#### 1. Create the table: sme_p_business_plan in Pre-Landed Layer

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import trim

In [0]:
def load_landed_internal_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

In [0]:
run_date = '2026-02-04'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = '`d-cdh`.landed'
cluster_columns = ["BusinessDate"]
landed_internal_table_path = f"{catalog_landed}.sme_business_plan"

In [0]:
# ============================== business_plan ============================

business_plan_extra_filters = None

if business_plan_extra_filters is not None:
    business_plan_extra_filters.append(date_filter)
    business_plan_final_filters = " AND ".join(
        business_plan_extra_filters
        )
else:
    business_plan_final_filters = date_filter

business_plan_selected_fields = [
            "ID",
            "DebtorCDI",
            "DebtorAFM",
            "DebtorBankGroupID",
            "IntentionDate",
            "CompletionTargetDate",
            "ApplicationDate",
            "Comments",
            "CreationDate",
            "CreationUser",
            "LastModifiedDate",
            "LastModifiedUser",
            "PortfolioID",
            "StatusID",
            "SolutionTypeID",
            "BusinessDate"
        ]

business_plan = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeBusinessPlan",
            business_plan_final_filters,
            business_plan_selected_fields,
        )

# ============================== bp_status ============================

bp_status_extra_filters = None

if bp_status_extra_filters is not None:
    bp_status_extra_filters.append(date_filter)
    bp_status_final_filters = " AND ".join(
        bp_status_extra_filters
        )
else:
    bp_status_final_filters = date_filter

bp_status_selected_fields = ["ID", "Label"]
bp_status = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpStatus",
            bp_status_final_filters,
            bp_status_selected_fields,
        )

# ============================== bp_solution_type ============================

bp_solution_type_extra_filters = None

if bp_solution_type_extra_filters is not None:
    bp_solution_type_extra_filters.append(date_filter)
    bp_solution_type_final_filters = " AND ".join(
        bp_solution_type_extra_filters
        )
else:
    bp_solution_type_final_filters = date_filter

bp_solution_type_selected_fields = ["ID", "Label"]
bp_solution_type = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpSolutionType",
            bp_solution_type_final_filters,
            bp_solution_type_selected_fields,
        )

# ============================== joins ============================

sme_business_plan = (
    business_plan.join(
        bp_status,
        business_plan["StatusID"] == bp_status["ID"],
        "left"
        )
        .join(
            bp_solution_type,
            business_plan["SolutionTypeID"] == bp_solution_type["ID"],
            "left"
            )
    )

# ============================== final landed - internal table ============================

sme_business_plan_df = sme_business_plan.select(
    business_plan.ID,
    business_plan.DebtorCDI,
    business_plan.DebtorAFM,
    business_plan.DebtorBankGroupID.alias("GroupCode"),
    bp_status.Label.alias("Business_plan_status"),
    bp_solution_type.Label.alias("SolutionType"),
    business_plan.IntentionDate.cast("Date"), 
    business_plan.CompletionTargetDate.cast("Date"),
    business_plan.ApplicationDate.cast("Date"),
    business_plan.Comments,
    business_plan.CreationDate.cast("timestamp"),
    business_plan.CreationUser,
    business_plan.LastModifiedDate.cast("timestamp"),
    business_plan.LastModifiedUser,
    business_plan.PortfolioID, 
    business_plan.BusinessDate
)
sme_business_plan_df.createOrReplaceTempView("sme_business_plan_df")